In [ ]:
pip install rouge-score sacrebleu evaluate torchsummary

In [ ]:
# ======================================================
# Cell 1: Imports & global setup
# ======================================================
import re
import warnings
from typing import Dict, Any

import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    T5Config,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

warnings.filterwarnings("ignore")

# Optional: set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ======================================================
# Cell 2: Load and inspect data
# ======================================================
df = pd.read_csv("/kaggle/input/layoutlm/medquad.csv")

print("Original Data Sample:")
print(df.head())
print("\nNull Value Data:")
print(df.isnull().sum())


In [ ]:
# ======================================================
# Cell 3: Basic filtering & cleaning
# ======================================================
question_words = [
    "what", "who", "why", "when", "where", "how",
    "is", "are", "does", "do", "can", "will", "shall"
]

# Lowercase questions for filtering
df["question"] = df["question"].str.lower()

# Keep only rows where question starts with a common question word
df = df[df["question"].str.split().str[0].isin(question_words)].reset_index(drop=True)

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Drop unused columns
df = df.drop(columns=["source", "focus_area"])

print("\nAfter dropping columns and duplicates:")
print(df.info())

# Remove duplicates based on question and answer
df = df.drop_duplicates(subset="question", keep="first").reset_index(drop=True)
df = df.drop_duplicates(subset="answer", keep="first").reset_index(drop=True)

# Drop rows with null question/answer
df = df.dropna(subset=["question", "answer"]).reset_index(drop=True)

# Ensure string type
df["question"] = df["question"].fillna("").astype(str)
df["answer"] = df["answer"].fillna("").astype(str)


def clean_text(text: str) -> str:
    """Remove parentheses content, extra spaces, lowercase."""
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\s+", " ", text.strip().lower())
    return text


# Apply cleaning
df["question"] = df["question"].apply(clean_text)
df["answer"] = df["answer"].apply(clean_text)

# Final normalization of spaces
df["question"] = df["question"].str.lower().str.strip().apply(
    lambda x: re.sub(r"\s+", " ", x)
)
df["answer"] = df["answer"].str.lower().str.strip().apply(
    lambda x: re.sub(r"\s+", " ", x)
)

print("\nNull Value Data After Cleaning:")
print(df.isnull().sum())
print(f"\nUnique questions: {df['question'].nunique()}")
print(f"Unique answers: {df['answer'].nunique()}")
print("\nFinal Data Sample:")
print(df.head())


In [ ]:
# ======================================================
# Cell 4: Model & tokenizer setup
# ======================================================
# You can change to "t5-small" if GPU memory is limited
model_name = "t5-base"

config = T5Config.from_pretrained(model_name)
config.dropout_rate = 0.1
config.feed_forward_proj = "gelu"

model = T5ForConditionalGeneration.from_pretrained(model_name, config=config)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Ensure embeddings size matches tokenizer vocab
model.resize_token_embeddings(len(tokenizer))


In [ ]:
# ======================================================
# Cell 5: Preprocessing function for seq2seq
# ======================================================
def preprocess_function(batch: Dict[str, Any]) -> Dict[str, Any]:
    """
    Tokenize questions and answers for seq2seq training.
    """
    inputs = [f"answer the following question: {q}" for q in batch["question"]]
    targets = [a for a in batch["answer"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=64,
            truncation=True,
            padding="max_length",
        )

    # Replace padding token id's in labels by -100 to ignore in loss
    labels_ids = np.array(labels["input_ids"])
    labels_ids = np.where(labels_ids == tokenizer.pad_token_id, -100, labels_ids)

    model_inputs["labels"] = labels_ids.tolist()
    return model_inputs

In [ ]:
# ======================================================
# Cell 6: Train / validation split & dataset mapping
# ======================================================
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=train_dataset.column_names,
    num_proc=4,
)

val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=val_dataset.column_names,
    num_proc=4,
)


In [ ]:
# ======================================================
# Cell 7: Training arguments
# ======================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_total_limit=2,
    learning_rate=5e-4,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    lr_scheduler_type="cosine_with_restarts",
    warmup_ratio=0.1,
    weight_decay=0.05,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    logging_steps=50,
    metric_for_best_model="exact_match",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=2,
    max_grad_norm=0.5,
    optim="adamw_torch_fused",
    generation_max_length=64,
    generation_num_beams=6,
    dataloader_num_workers=4,
    group_by_length=True,
    remove_unused_columns=True,
    label_smoothing_factor=0.1,
)



In [ ]:
# ======================================================
# Cell 8: Data collator & metrics
# ======================================================
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    return_tensors="pt",
)

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")


def compute_metrics(eval_pred):
    """
    Compute Exact Match, BLEU, and ROUGE-L.
    """
    predictions, labels = eval_pred

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels back to pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [t.strip().lower() for t in decoded_preds]
    decoded_labels = [t.strip().lower() for t in decoded_labels]

    # Exact match
    exact_match = float(
        np.mean([p == l for p, l in zip(decoded_preds, decoded_labels)])
    )

    # BLEU
    bleu_score = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
    )["bleu"]

    # ROUGE-L
    rouge_score = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
    )["rougeL"]

    return {
        "exact_match": exact_match,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
    }



In [ ]:
# ======================================================
# Cell 9: Trainer, training, and saving
# ======================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save model & tokenizer
trainer.save_model("./t5_chatbot_model")
tokenizer.save_pretrained("./t5_chatbot_tokenizer")


In [ ]:
# ======================================================
# Cell 10: Inference / Chatbot testing
# ======================================================
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load tokenizer and model from saved paths
inference_model_path = "./t5_chatbot_model"
inference_tokenizer_path = "./t5_chatbot_tokenizer"

inference_tokenizer = T5Tokenizer.from_pretrained(inference_tokenizer_path)
inference_model = T5ForConditionalGeneration.from_pretrained(inference_model_path)

inference_model.eval()


def generate_response(question: str, max_length: int = 64, num_beams: int = 5) -> str:
    """
    Generate chatbot response using beam search.
    """
    formatted_question = f"answer the following question: {question}"

    inputs = inference_tokenizer(
        formatted_question,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True,
    )

    with torch.no_grad():
        outputs = inference_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=inference_tokenizer.pad_token_id,
        )

    response = inference_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Example test
test_question = "What is Alzheimer's disease?"
test_response = generate_response(test_question)

print("\n=== Chatbot Test ===")
print("Question:", test_question)
print("Bot Response:", test_response)

In [ ]:
# Define a question to ask the model
question = "I had a surgery which ended up with some failures. What can I do to fix it?"

# Generate a response using the `generate_response_top_k_top_p` function
response = generate_response(question)

# Print the question and the generated response
print("Question:", question)
print("Response:", response)